# TAMP-OS Batch Lithograph Maker v2

Select multiple microscopy images and convert them all to STL/3MF/GLB.

**What's new in v2:**
- **Low / Medium / High quality presets** calculated automatically from your printer's nozzle diameter and layer height — no need to guess resolution or blur values
- **Relief height** now shows how many distinct height levels your printer can actually produce
- **Full customization panel** still available for advanced users

> ⚠️ Run in **Jupyter Lab or Jupyter Notebook** — not VS Code notebook viewer (tkinter needs a real display).

### Step 1 — Install dependencies (once)

In [5]:
!pip install numpy pillow scipy numpy-stl

### Step 2 — Imports

In [6]:
import os, sys, struct, json, zipfile, math
from pathlib import Path
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter
from stl import mesh
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import threading
print("Imports OK")

Imports OK


### Step 3 — Pipeline functions & preset calculator
*(self-contained — no external files needed)*

In [7]:
# ── Preset calculator ─────────────────────────────────────────────────────────

def calculate_presets(print_size_x, nozzle_mm=0.4, layer_mm=0.12):
    """
    Calculate Low / Medium / High presets from printer physical specs.

    The key insight:
      - 1 pixel on the height map = 1 physical feature on the print
      - The smallest printable feature = nozzle diameter
      - So: optimal resolution = print_width / nozzle_diameter

    Medium = nozzle-matched (1 px per nozzle width) — the sweet spot
    Low    = 2x nozzle width (coarser, much smaller file)
    High   = 0.5x nozzle width (finer than nozzle, captures more texture)
    """
    medium_res = round(print_size_x / nozzle_mm)
    low_res    = round(print_size_x / (nozzle_mm * 2))
    high_res   = round(print_size_x / (nozzle_mm * 0.5))

    def snap(v): return max(64, min(512, int(2 ** round(math.log2(v)))))
    low_res    = snap(low_res)
    medium_res = snap(medium_res)
    high_res   = min(512, snap(high_res))

    px_size_low    = print_size_x / low_res
    px_size_medium = print_size_x / medium_res
    px_size_high   = print_size_x / high_res

    return {
        "Low": {
            "resolution": low_res,
            "blur": 2.0,
            "px_size": px_size_low,
            "description": (f"{low_res} px  |  1 px = {px_size_low:.2f} mm  |  "
                            f"~{nozzle_mm*2:.1f} mm features  |  smooth, small file")
        },
        "Medium": {
            "resolution": medium_res,
            "blur": 1.2,
            "px_size": px_size_medium,
            "description": (f"{medium_res} px  |  1 px = {px_size_medium:.2f} mm  |  "
                            f"~{nozzle_mm:.1f} mm features  |  matches nozzle exactly")
        },
        "High": {
            "resolution": high_res,
            "blur": 0.8,
            "px_size": px_size_high,
            "description": (f"{high_res} px  |  1 px = {px_size_high:.2f} mm  |  "
                            f"~{nozzle_mm*0.5:.2f} mm features  |  finer than nozzle")
        },
    }

# ── Height map ────────────────────────────────────────────────────────────────

BEST_PRACTICE = {"base_thickness": 1.0, "relief_height": 3.0}

def smart_size_y(img_path, size_x=100.0):
    try:
        img = Image.open(img_path); w, h = img.size
        return round(size_x * h / w, 1)
    except Exception: return 75.0

def image_to_heightmap(image_path, output_size=(512,512), invert=False,
                       blur_sigma=1.0, contrast_percentile=(2.0,98.0),
                       preserve_aspect=True, flip=True):
    img = Image.open(image_path).convert("L")
    orig_w, orig_h = img.size
    if preserve_aspect and orig_w != orig_h:
        target_w = output_size[0]
        target_h = round(target_w * orig_h / orig_w)
        actual_size = (target_w, target_h)
        print(f"  [i] {orig_w}x{orig_h} -> {target_w}x{target_h}")
    else:
        actual_size = output_size
    img = img.resize(actual_size, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32)
    lo = np.percentile(arr, contrast_percentile[0])
    hi = np.percentile(arr, contrast_percentile[1])
    arr = np.clip((arr - lo) / (hi - lo + 1e-8), 0.0, 1.0)
    if blur_sigma > 0:
        arr = gaussian_filter(arr, sigma=blur_sigma)
    if invert:
        arr = 1.0 - arr
    if flip:
        arr = np.flipud(arr)
    return arr

def heightmap_to_stl(heightmap, output_path, base_thickness_mm=1.0,
                     max_relief_mm=3.0, physical_size_mm=(100.0,100.0)):
    rows, cols = heightmap.shape
    dx = physical_size_mm[0] / (cols - 1)
    dy = physical_size_mm[1] / (rows - 1)
    hm_ratio = cols / rows
    print_ratio = physical_size_mm[0] / physical_size_mm[1]
    if abs(hm_ratio - print_ratio) > 0.05:
        print(f"  [WARNING] Aspect mismatch - consider size_y={round(physical_size_mm[0]/hm_ratio,1)}")
    z_top = base_thickness_mm + heightmap * max_relief_mm
    num_tris = (rows-1)*(cols-1)*4 + 2*((rows-1)+(cols-1))*2
    litho_mesh = mesh.Mesh(np.zeros(num_tris, dtype=mesh.Mesh.dtype))
    tri_idx = 0
    def add_tri(v0, v1, v2):
        nonlocal tri_idx
        litho_mesh.vectors[tri_idx] = [v0, v1, v2]
        tri_idx += 1
    for r in range(rows-1):
        for c in range(cols-1):
            x0,y0 = c*dx, r*dy
            x1 = (c+1)*dx
            x2,y2 = c*dx, (r+1)*dy
            x3,y3 = (c+1)*dx, (r+1)*dy
            z00,z10,z01,z11 = z_top[r,c],z_top[r,c+1],z_top[r+1,c],z_top[r+1,c+1]
            add_tri([x0,y0,z00],[x1,y0,z10],[x2,y2,z01])
            add_tri([x1,y0,z10],[x3,y3,z11],[x2,y2,z01])
            add_tri([x0,y0,0],[x2,y2,0],[x1,y0,0])
            add_tri([x1,y0,0],[x2,y2,0],[x3,y3,0])
    xmax,ymax = (cols-1)*dx,(rows-1)*dy
    for c in range(cols-1):
        x0,x1 = c*dx,(c+1)*dx
        z0,z1 = z_top[0,c],z_top[0,c+1]
        add_tri([x0,0,0],[x1,0,0],[x0,0,z0])
        add_tri([x1,0,0],[x1,0,z1],[x0,0,z0])
    for c in range(cols-1):
        x0,x1 = c*dx,(c+1)*dx
        z0,z1 = z_top[-1,c],z_top[-1,c+1]
        add_tri([x0,ymax,0],[x0,ymax,z0],[x1,ymax,0])
        add_tri([x1,ymax,0],[x0,ymax,z0],[x1,ymax,z1])
    for r in range(rows-1):
        y0,y1 = r*dy,(r+1)*dy
        z0,z1 = z_top[r,0],z_top[r+1,0]
        add_tri([0,y0,0],[0,y0,z0],[0,y1,0])
        add_tri([0,y1,0],[0,y0,z0],[0,y1,z1])
    for r in range(rows-1):
        y0,y1 = r*dy,(r+1)*dy
        z0,z1 = z_top[r,-1],z_top[r+1,-1]
        add_tri([xmax,y0,0],[xmax,y1,0],[xmax,y0,z0])
        add_tri([xmax,y1,0],[xmax,y1,z1],[xmax,y0,z0])
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    litho_mesh.save(str(output_path))
    print(f"  [OK] STL -> {output_path}")
    return Path(output_path)

# ── Format converters ─────────────────────────────────────────────────────────

def stl_to_3mf(stl_path):
    from stl import mesh as stl_mesh
    m = stl_mesh.Mesh.from_file(str(stl_path))
    vertices = []; vert_map = {}; indices = []
    for tri in m.vectors:
        face = []
        for v in tri:
            key = tuple(float(x) for x in v)
            if key not in vert_map:
                vert_map[key] = len(vertices)
                vertices.append(key)
            face.append(vert_map[key])
        indices.append(face)
    vx = " ".join(
        f'<vertex x="{v[0]:.4f}" y="{v[1]:.4f}" z="{v[2]:.4f}"/>' for v in vertices)
    tx = " ".join(
        f'<triangle v1="{t[0]}" v2="{t[1]}" v3="{t[2]}"/>' for t in indices)
    ns = "http://schemas.microsoft.com/3dmanufacturing/core/2015/02"
    xml = (f'<?xml version="1.0" encoding="UTF-8"?>'
           f'<model unit="millimeter" xmlns="{ns}">'
           f'<resources><object id="1" type="model"><mesh>'
           f'<vertices>{vx}</vertices><triangles>{tx}</triangles>'
           f'</mesh></object></resources><build><item objectid="1"/></build></model>')
    out = Path(str(stl_path).replace(".stl", ".3mf"))
    ct_ns = "http://schemas.openxmlformats.org/package/2006/content-types"
    rel_ns = "http://schemas.openxmlformats.org/package/2006/relationships"
    rel_type = "http://schemas.microsoft.com/3dmanufacturing/2013/01/3dmodel"
    with zipfile.ZipFile(str(out), "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("3D/3dmodel.model", xml)
        zf.writestr("[Content_Types].xml",
            f'<?xml version="1.0"?><Types xmlns="{ct_ns}">'
            f'<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>'
            f'<Default Extension="model" ContentType="application/vnd.ms-package.3dmanufacturing-3dmodel+xml"/>'
            f'</Types>')
        zf.writestr("_rels/.rels",
            f'<?xml version="1.0"?><Relationships xmlns="{rel_ns}">'
            f'<Relationship Type="{rel_type}" Target="/3D/3dmodel.model" Id="r1"/>'
            f'</Relationships>')
    print(f"  [OK] 3MF -> {out}")
    return out

def stl_to_glb(stl_path):
    from stl import mesh as stl_mesh
    m = stl_mesh.Mesh.from_file(str(stl_path))
    verts = m.vectors.reshape(-1, 3).astype(np.float32)
    indices = np.arange(len(verts), dtype=np.uint32)
    def pad4(b): return b + b'\x00' * ((4 - len(b) % 4) % 4)
    ib = pad4(indices.tobytes())
    vb = pad4(verts.tobytes())
    bd = ib + vb
    gltf = {
        "asset": {"version": "2.0", "generator": "TAMP-OS"}, "scene": 0,
        "scenes": [{"nodes": [0]}], "nodes": [{"mesh": 0}],
        "meshes": [{"primitives": [{"attributes": {"POSITION": 1}, "indices": 0}]}],
        "accessors": [
            {"bufferView": 0, "componentType": 5125, "count": len(indices), "type": "SCALAR"},
            {"bufferView": 1, "componentType": 5126, "count": len(verts), "type": "VEC3",
             "min": verts.min(0).tolist(), "max": verts.max(0).tolist()}],
        "bufferViews": [
            {"buffer": 0, "byteOffset": 0, "byteLength": len(ib), "target": 34963},
            {"buffer": 0, "byteOffset": len(ib), "byteLength": len(vb), "target": 34962}],
        "buffers": [{"byteLength": len(bd)}]
    }
    jb = pad4(json.dumps(gltf, separators=(",", ":")).encode())
    def chunk(d, t): return struct.pack("<II", len(d), t) + d
    out = Path(str(stl_path).replace(".stl", ".glb"))
    with open(str(out), "wb") as f:
        f.write(struct.pack("<III", 0x46546C67, 2, 12 + 8 + len(jb) + 8 + len(bd))
                + chunk(jb, 0x4E4F534A) + chunk(bd, 0x004E4942))
    print(f"  [OK] GLB -> {out}")
    return out

def convert_stl(stl_path, formats):
    if formats.get("3mf"): stl_to_3mf(stl_path)
    if formats.get("glb"):  stl_to_glb(stl_path)
    if not formats.get("stl"): stl_path.unlink(missing_ok=True)

print("All pipeline functions loaded OK.")

All pipeline functions loaded OK.


## Step 4 — Launch the GUI
Run this cell to open the window.

In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import threading
import math

# ── Preset calculator ─────────────────────────────────────────────────────────

def calculate_presets(print_size_x, nozzle_mm=0.4, layer_mm=0.12):
    medium_res = round(print_size_x / nozzle_mm)
    low_res    = round(print_size_x / (nozzle_mm * 2))
    high_res   = round(print_size_x / (nozzle_mm * 0.5))
    def snap(v): return max(64, min(512, int(2 ** round(math.log2(v)))))
    low_res    = snap(low_res)
    medium_res = snap(medium_res)
    high_res   = min(512, snap(high_res))
    px_low     = print_size_x / low_res
    px_med     = print_size_x / medium_res
    px_high    = print_size_x / high_res
    return {
        "Low": {
            "resolution": low_res, "blur": 2.0, "px_size": px_low,
            "description": (f"{low_res} px  |  1 px = {px_low:.2f} mm  |  "
                            f"~{nozzle_mm*2:.1f} mm features  |  smooth, small file")
        },
        "Medium": {
            "resolution": medium_res, "blur": 1.2, "px_size": px_med,
            "description": (f"{medium_res} px  |  1 px = {px_med:.2f} mm  |  "
                            f"~{nozzle_mm:.1f} mm features  |  matches nozzle exactly")
        },
        "High": {
            "resolution": high_res, "blur": 0.8, "px_size": px_high,
            "description": (f"{high_res} px  |  1 px = {px_high:.2f} mm  |  "
                            f"~{nozzle_mm*0.5:.2f} mm features  |  finer than nozzle")
        },
    }

BEST_PRACTICE = {"base_thickness": 1.0, "relief_height": 3.0}

def smart_size_y(img_path, size_x=100.0):
    try:
        img = Image.open(img_path); w, h = img.size
        return round(size_x * h / w, 1)
    except Exception: return 75.0


# ── Scrollable base window ────────────────────────────────────────────────────

class ScrollableFrame(tk.Frame):
    """A frame that becomes scrollable when content exceeds screen height."""
    def __init__(self, parent):
        super().__init__(parent)
        self.canvas = tk.Canvas(self, borderwidth=0, highlightthickness=0)
        self.scrollbar = tk.Scrollbar(self, orient="vertical",
                                      command=self.canvas.yview)
        self.inner = tk.Frame(self.canvas)

        self.canvas.configure(yscrollcommand=self.scrollbar.set)
        self.scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        self.window_id = self.canvas.create_window(
            (0, 0), window=self.inner, anchor="nw")

        self.inner.bind("<Configure>", self._on_frame_configure)
        self.canvas.bind("<Configure>", self._on_canvas_configure)

        # Mouse wheel scrolling
        self.canvas.bind_all("<MouseWheel>",
            lambda e: self.canvas.yview_scroll(-1*(e.delta//120), "units"))
        self.canvas.bind_all("<Button-4>",
            lambda e: self.canvas.yview_scroll(-1, "units"))
        self.canvas.bind_all("<Button-5>",
            lambda e: self.canvas.yview_scroll(1, "units"))

    def _on_frame_configure(self, event):
        self.canvas.configure(scrollregion=self.canvas.bbox("all"))

    def _on_canvas_configure(self, event):
        self.canvas.itemconfig(self.window_id, width=event.width)


# ── Main App ──────────────────────────────────────────────────────────────────

class TAMPBatchGUIv2:
    def __init__(self, root):
        self.root = root
        self.root.title("TAMP-OS Batch Lithograph Maker v2")

        # Set window size to 90% of screen height
        sw = root.winfo_screenwidth()
        sh = root.winfo_screenheight()
        win_w = 660
        win_h = min(900, int(sh * 0.88))
        root.geometry(f"{win_w}x{win_h}")
        root.resizable(False, True)

        # Scrollable container
        self.sf = ScrollableFrame(root)
        self.sf.pack(fill=tk.BOTH, expand=True)
        self.f = self.sf.inner   # all widgets go here

        self.image_paths = []
        self.running = False
        self.advanced_open = tk.BooleanVar(value=False)
        self.selected_preset = tk.StringVar(value="Medium")
        self._build_ui()
        self._update_preset_display()

    def _build_ui(self):
        f = self.f
        pad = dict(padx=10, pady=4)

        # Title
        tk.Label(f, text="TAMP-OS Batch Lithograph Maker",
                 font=("Helvetica", 14, "bold")).grid(
            row=0, column=0, columnspan=3, pady=(14,2))
        tk.Label(f, text="Select microscopy images and convert them all to STL in one go.",
                 font=("Helvetica", 9), fg="gray").grid(
            row=1, column=0, columnspan=3, pady=(0,8))

        # Image selection
        tk.Label(f, text="Images:",
                 font=("Helvetica", 10, "bold")).grid(row=2, column=0, sticky="nw", **pad)
        fl = tk.Frame(f)
        fl.grid(row=2, column=1, columnspan=2, **pad)
        sb = tk.Scrollbar(fl, orient=tk.VERTICAL)
        self.listbox = tk.Listbox(fl, width=48, height=5,
                                  yscrollcommand=sb.set, selectmode=tk.EXTENDED)
        sb.config(command=self.listbox.yview)
        self.listbox.pack(side=tk.LEFT, fill=tk.BOTH)
        sb.pack(side=tk.RIGHT, fill=tk.Y)

        bf = tk.Frame(f)
        bf.grid(row=3, column=1, columnspan=2, sticky="w", padx=10)
        tk.Button(bf, text="+ Add Images",
                  command=self._add_images, width=14).pack(side=tk.LEFT, padx=(0,5))
        tk.Button(bf, text="- Remove Selected",
                  command=self._remove_selected, width=14).pack(side=tk.LEFT)

        # Output folder
        tk.Label(f, text="Output folder:",
                 font=("Helvetica", 10, "bold")).grid(row=4, column=0, sticky="w", **pad)
        self.output_var = tk.StringVar(value=str(Path.home() / "TAMP_output"))
        tk.Entry(f, textvariable=self.output_var, width=36).grid(
            row=4, column=1, sticky="w", **pad)
        tk.Button(f, text="Browse",
                  command=self._browse_output).grid(row=4, column=2, sticky="w", padx=(0,10))

        # Output format
        tk.Label(f, text="Output format:",
                 font=("Helvetica", 10, "bold")).grid(row=5, column=0, sticky="w", **pad)
        ff = tk.Frame(f)
        ff.grid(row=5, column=1, columnspan=2, sticky="w", padx=10)
        self.fmt_stl = tk.BooleanVar(value=True)
        self.fmt_3mf = tk.BooleanVar(value=False)
        self.fmt_glb = tk.BooleanVar(value=False)
        tk.Checkbutton(ff, text=".STL", variable=self.fmt_stl).pack(side=tk.LEFT, padx=(0,8))
        tk.Checkbutton(ff, text=".3MF", variable=self.fmt_3mf).pack(side=tk.LEFT, padx=(0,8))
        tk.Checkbutton(ff, text=".GLB", variable=self.fmt_glb).pack(side=tk.LEFT, padx=(0,8))
        tk.Label(ff, text="(STL=universal, 3MF=PrusaSlicer, GLB=web)",
                 fg="gray", font=("Helvetica", 8)).pack(side=tk.LEFT)

        # Flip / Invert
        ttk.Separator(f, orient="horizontal").grid(
            row=6, column=0, columnspan=3, sticky="ew", padx=10, pady=5)
        self.flip_var   = tk.BooleanVar(value=True)
        self.invert_var = tk.BooleanVar(value=False)
        tk.Checkbutton(f, text="Flip vertically (recommended - matches image orientation)",
                       variable=self.flip_var).grid(
            row=7, column=0, columnspan=3, sticky="w", padx=10, pady=2)
        tk.Checkbutton(f, text="Invert relief (dark areas raised - for some TEM images)",
                       variable=self.invert_var).grid(
            row=8, column=0, columnspan=3, sticky="w", padx=10, pady=2)

        # ── PRINT QUALITY PRESETS ─────────────────────────────────────────────
        ttk.Separator(f, orient="horizontal").grid(
            row=9, column=0, columnspan=3, sticky="ew", padx=10, pady=5)
        tk.Label(f, text="Print Quality",
                 font=("Helvetica", 11, "bold")).grid(
            row=10, column=0, columnspan=3, pady=(0,2))
        tk.Label(f, text="Resolution and blur are calculated automatically from your printer specs.",
                 font=("Helvetica", 9), fg="gray").grid(
            row=11, column=0, columnspan=3, pady=(0,4))

        preset_frame = tk.LabelFrame(f, text="Printer specs + quality preset",
                                     font=("Helvetica", 9), padx=8, pady=6)
        preset_frame.grid(row=12, column=0, columnspan=3, padx=14, pady=(0,4), sticky="ew")

        # Printer spec inputs
        spec_row = tk.Frame(preset_frame)
        spec_row.grid(row=0, column=0, columnspan=3, sticky="w", pady=(0,6))
        tk.Label(spec_row, text="Print width (mm):").pack(side=tk.LEFT)
        self.size_x_var = tk.StringVar(value="100")
        sx = tk.Entry(spec_row, textvariable=self.size_x_var, width=6)
        sx.pack(side=tk.LEFT, padx=(4,14))
        sx.bind("<FocusOut>", lambda e: self._update_preset_display())
        tk.Label(spec_row, text="Nozzle (mm):").pack(side=tk.LEFT)
        self.nozzle_var = tk.StringVar(value="0.4")
        nz = tk.Entry(spec_row, textvariable=self.nozzle_var, width=5)
        nz.pack(side=tk.LEFT, padx=(4,14))
        nz.bind("<FocusOut>", lambda e: self._update_preset_display())
        tk.Label(spec_row, text="Layer height (mm):").pack(side=tk.LEFT)
        self.layer_var = tk.StringVar(value="0.12")
        lh = tk.Entry(spec_row, textvariable=self.layer_var, width=5)
        lh.pack(side=tk.LEFT, padx=(4,0))
        lh.bind("<FocusOut>", lambda e: self._update_preset_display())

        # Preset cards
        self.preset_descriptions = {}
        colors = {"Low": "#e8f4e8", "Medium": "#e8f0ff", "High": "#fff0e8"}
        for col, preset in enumerate(["Low", "Medium", "High"]):
            pf = tk.Frame(preset_frame, relief="groove", bd=1, bg=colors[preset])
            pf.grid(row=1, column=col, padx=5, pady=4, sticky="nsew")
            preset_frame.columnconfigure(col, weight=1)
            tk.Radiobutton(pf, text=preset, variable=self.selected_preset,
                           value=preset, font=("Helvetica", 11, "bold"),
                           bg=colors[preset],
                           command=self._update_preset_display).pack(
                fill=tk.X, padx=4, pady=(6,2))
            desc_var = tk.StringVar(value="")
            self.preset_descriptions[preset] = desc_var
            tk.Label(pf, textvariable=desc_var, fg="gray",
                     font=("Helvetica", 8), justify="left",
                     wraplength=150, bg=colors[preset]).pack(
                padx=6, pady=(0,6), fill=tk.X)

        self.preset_calc_var = tk.StringVar(value="")
        tk.Label(preset_frame, textvariable=self.preset_calc_var,
                 fg="#2a7ae2", font=("Helvetica", 9), justify="left").grid(
            row=2, column=0, columnspan=3, sticky="w", pady=(4,0))

        # Print height
        sy_row = tk.Frame(preset_frame)
        sy_row.grid(row=3, column=0, columnspan=3, sticky="w", pady=(4,0))
        tk.Label(sy_row, text="Print height (mm):").pack(side=tk.LEFT)
        self.size_y_var = tk.StringVar(value="auto")
        tk.Entry(sy_row, textvariable=self.size_y_var, width=7).pack(
            side=tk.LEFT, padx=(4,8))
        tk.Label(sy_row, text="auto = calculated from image aspect ratio",
                 fg="gray", font=("Helvetica", 8)).pack(side=tk.LEFT)

        # Relief height
        rh_row = tk.Frame(preset_frame)
        rh_row.grid(row=4, column=0, columnspan=3, sticky="w", pady=(4,4))
        tk.Label(rh_row, text="Relief height (mm):").pack(side=tk.LEFT)
        self.relief_var = tk.StringVar(value="3.0")
        tk.Entry(rh_row, textvariable=self.relief_var, width=6).pack(
            side=tk.LEFT, padx=(4,8))
        self.relief_levels_var = tk.StringVar(value="")
        tk.Label(rh_row, textvariable=self.relief_levels_var,
                 fg="gray", font=("Helvetica", 8)).pack(side=tk.LEFT)

        # ── ADVANCED TOGGLE ───────────────────────────────────────────────────
        ttk.Separator(f, orient="horizontal").grid(
            row=13, column=0, columnspan=3, sticky="ew", padx=10, pady=5)
        adv_hdr = tk.Frame(f)
        adv_hdr.grid(row=14, column=0, columnspan=3, padx=14, pady=(0,2), sticky="w")
        tk.Checkbutton(adv_hdr, text="Full customization",
                       variable=self.advanced_open,
                       font=("Helvetica", 10, "bold"),
                       command=self._toggle_advanced).pack(side=tk.LEFT)
        tk.Label(adv_hdr,
                 text="Warning: override calculated values - uncharted territory",
                 fg="#b05000", font=("Helvetica", 8)).pack(side=tk.LEFT, padx=6)

        self.adv_panel = tk.LabelFrame(f, text="Advanced parameters",
                                       font=("Helvetica", 9), padx=8, pady=6)
        adv_params = [
            ("Resolution (px)",     "resolution",    "256", "Overrides preset"),
            ("Blur (sigma)",        "blur",          "1.2",  "Overrides preset"),
            ("Base thickness (mm)", "base_thickness","1.0",  "Solid base below relief"),
        ]
        self.adv_vars = {}
        for i, (label, key, default, hint) in enumerate(adv_params):
            tk.Label(self.adv_panel, text=label+":").grid(
                row=i, column=0, sticky="w", pady=2)
            var = tk.StringVar(value=default)
            self.adv_vars[key] = var
            tk.Entry(self.adv_panel, textvariable=var, width=10).grid(
                row=i, column=1, sticky="w", padx=6, pady=2)
            tk.Label(self.adv_panel, text=hint, fg="gray",
                     font=("Helvetica", 8)).grid(
                row=i, column=2, sticky="w", padx=(0,8), pady=2)

        # Progress
        ttk.Separator(f, orient="horizontal").grid(
            row=16, column=0, columnspan=3, sticky="ew", padx=10, pady=5)
        self.progress = ttk.Progressbar(f, orient=tk.HORIZONTAL,
                                        length=400, mode="determinate")
        self.progress.grid(row=17, column=0, columnspan=3, padx=10, pady=4)
        self.status_var = tk.StringVar(value="Ready.")
        tk.Label(f, textvariable=self.status_var, fg="gray",
                 font=("Helvetica", 9)).grid(
            row=18, column=0, columnspan=3, pady=(0,4))
        self.run_btn = tk.Button(f, text="Generate files",
                                 font=("Helvetica", 11, "bold"),
                                 bg="#2a7ae2", fg="white",
                                 activebackground="#1a5fbf", activeforeground="white",
                                 command=self._run, width=20, height=2)
        self.run_btn.grid(row=19, column=0, columnspan=3, pady=(4,14))

    # ── Preset logic ──────────────────────────────────────────────────────────

    def _update_preset_display(self):
        try: size_x = float(self.size_x_var.get())
        except Exception: size_x = 100.0
        try: nozzle = float(self.nozzle_var.get())
        except Exception: nozzle = 0.4
        try: layer  = float(self.layer_var.get())
        except Exception: layer = 0.12
        try: relief = float(self.relief_var.get())
        except Exception: relief = 3.0

        presets = calculate_presets(size_x, nozzle, layer)
        for name, data in presets.items():
            self.preset_descriptions[name].set(data["description"])

        chosen = self.selected_preset.get()
        p = presets[chosen]
        self.preset_calc_var.set(
            f"Selected: resolution = {p['resolution']} px, blur = {p['blur']}  "
            f"(1 px = {size_x/p['resolution']:.2f} mm on print)")
        levels = round(relief / layer)
        self.relief_levels_var.set(
            f"= {levels} distinct height levels at {layer} mm layer height")
        if not self.advanced_open.get():
            self.adv_vars["resolution"].set(str(p["resolution"]))
            self.adv_vars["blur"].set(str(p["blur"]))

    def _toggle_advanced(self):
        if self.advanced_open.get():
            self.adv_panel.grid(row=15, column=0, columnspan=3,
                                padx=14, pady=(0,4), sticky="ew")
        else:
            self.adv_panel.grid_remove()
            self._update_preset_display()

    # ── File helpers ──────────────────────────────────────────────────────────

    def _add_images(self):
        files = filedialog.askopenfilenames(
            title="Select images",
            filetypes=[("Image files", "*.png *.jpg *.jpeg *.tif *.tiff *.bmp"),
                       ("All files", "*.*")])
        for f in files:
            if f not in self.image_paths:
                self.image_paths.append(f)
                self.listbox.insert(tk.END, Path(f).name)

    def _remove_selected(self):
        for i in reversed(list(self.listbox.curselection())):
            self.listbox.delete(i)
            self.image_paths.pop(i)

    def _browse_output(self):
        folder = filedialog.askdirectory()
        if folder: self.output_var.set(folder)

    def _get_params(self, img_path):
        try: size_x = float(self.size_x_var.get())
        except Exception: size_x = 100.0
        try: size_y = float(self.size_y_var.get())
        except Exception: size_y = smart_size_y(img_path, size_x)
        try: relief = float(self.relief_var.get())
        except Exception: relief = 3.0
        try: nozzle = float(self.nozzle_var.get())
        except Exception: nozzle = 0.4
        try: layer  = float(self.layer_var.get())
        except Exception: layer = 0.12

        presets = calculate_presets(size_x, nozzle, layer)
        p = presets[self.selected_preset.get()]

        if self.advanced_open.get():
            try: resolution = int(self.adv_vars["resolution"].get())
            except Exception: resolution = p["resolution"]
            try: blur = float(self.adv_vars["blur"].get())
            except Exception: blur = p["blur"]
            try: base = float(self.adv_vars["base_thickness"].get())
            except Exception: base = BEST_PRACTICE["base_thickness"]
        else:
            resolution = p["resolution"]
            blur       = p["blur"]
            base       = BEST_PRACTICE["base_thickness"]

        return {"size_x": size_x, "size_y": size_y, "relief_height": relief,
                "base_thickness": base, "blur": blur, "resolution": resolution,
                "invert": self.invert_var.get(), "flip": self.flip_var.get()}

    def _get_formats(self):
        fmt = {"stl": self.fmt_stl.get(), "3mf": self.fmt_3mf.get(),
               "glb": self.fmt_glb.get()}
        if not any(fmt.values()):
            messagebox.showwarning("No format", "Select at least one output format.")
            return None
        return fmt

    # ── Pipeline ─────────────────────────────────────────────────────────────

    def _run(self):
        if not self.image_paths:
            messagebox.showwarning("No images", "Add at least one image.")
            return
        formats = self._get_formats()
        if formats is None or self.running: return
        self.running = True
        self.run_btn.config(state=tk.DISABLED)
        threading.Thread(target=self._process_all, args=(formats,),
                         daemon=True).start()

    def _process_all(self, formats):
        output_dir = Path(self.output_var.get())
        output_dir.mkdir(parents=True, exist_ok=True)
        total = len(self.image_paths)
        errors = []
        for i, img_path in enumerate(self.image_paths):
            name = Path(img_path).stem
            self._set_status(f"Processing {i+1}/{total}: {name}...")
            self.progress["value"] = (i / total) * 100
            self.root.update_idletasks()
            try:
                params = self._get_params(img_path)
                hm = image_to_heightmap(
                    img_path,
                    output_size=(params["resolution"], params["resolution"]),
                    invert=params["invert"], blur_sigma=params["blur"],
                    preserve_aspect=True, flip=params["flip"])
                stl_path = output_dir / f"{name}_lithograph.stl"
                heightmap_to_stl(hm, stl_path,
                    base_thickness_mm=params["base_thickness"],
                    max_relief_mm=params["relief_height"],
                    physical_size_mm=(params["size_x"], params["size_y"]))
                convert_stl(stl_path, formats)
            except Exception as e:
                errors.append(f"{name}: {e}")
        self.progress["value"] = 100
        fmt_list = ", ".join(f".{k.upper()}" for k, v in formats.items() if v)
        if errors:
            self._set_status(f"Done with {len(errors)} error(s).")
            messagebox.showerror("Errors", "Some files failed:\n\n" + "\n".join(errors))
        else:
            self._set_status(f"Done! {total} file(s) as {fmt_list}")
            messagebox.showinfo("Done!",
                f"Generated {total} file(s) as {fmt_list}.\n\nSaved to:\n{output_dir}")
        self.running = False
        self.run_btn.config(state=tk.NORMAL)

    def _set_status(self, msg):
        self.status_var.set(msg)
        self.root.update_idletasks()


# Launch
root = tk.Tk()
app = TAMPBatchGUIv2(root)
root.mainloop()


  [i] 3717x3712 -> 128x128
  [OK] STL -> C:\Users\vazquez\TAMP_output\Asian_elephant_trunk_HeidelbergZoo_1_lithograph.stl
  [i] 3717x3712 -> 256x256
  [OK] STL -> C:\Users\vazquez\TAMP_output\Asian_elephant_trunk_HeidelbergZoo_1_lithograph.stl
  [i] 3717x3712 -> 512x511
  [OK] STL -> C:\Users\vazquez\TAMP_output\Asian_elephant_trunk_HeidelbergZoo_1_lithograph.stl
